In [1]:
import xarray as xr
import pandas as pd
import sqlalchemy
import pygrib
import sqlite3

import numpy as np
from pathlib import Path

import numpy as np



1. Get ERA5 data with era5Get.py
2. Starts with 2t,2d,pcp on a .10 degree grid (2981 points total)
3. data is 3 hourly 
4. grib2db.ipynb -> /home/joe/work/Fire/ML/New/DB/era5_daily_2982_{TARGET_VAR}.sqlite
6. 
8. 
9. Monthly means are then computed by variable  -> /home/joe/work/Fire/ML/Data/DB/era5_means_2982_{TARGET_VAR}.sqlite
10. Monthly means combined into 1 table, and RH and VPD are computed

## Compute Monthly Means

### Compute Daily Means 

In [8]:
import sqlite3
from pathlib import Path
import pandas as pd
TARGET_VAR = 'tp'
sqlite_path = Path(f"/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_{TARGET_VAR}.sqlite")
table_name = "daily_data"
latlons = pd.read_csv("latlons_42x71.csv")
sqlite_path_out = Path(f"/home/joe/work/Fire/ML/Data/DB/era5_means_2982_{TARGET_VAR}.sqlite")
table_name_out = "monthly_means"

create_sql = f"""
CREATE TABLE IF NOT EXISTS {table_name_out} (
    point_id INTEGER NOT NULL,
    latitude REAL NOT NULL,
    longitude REAL NOT NULL,
    var TEXT NOT NULL,
    yrmo DATE NOT NULL,
    value REAL,
    PRIMARY KEY (yrmo, point_id, var)
);
"""

## Create table if it does not exist
with sqlite3.connect(sqlite_path_out) as conn:
    conn.execute(create_sql)



In [9]:
if sqlite_path.exists():
    with sqlite3.connect(sqlite_path_out) as conn:
        max_yrmo = pd.read_sql_query(f"SELECT max(yrmo) FROM {table_name_out}", conn).values[0][0]
        max_yrmo = int(max_yrmo)*100+31
        print(f"Loaded max yrmo: {max_yrmo} from {sqlite_path}")
else:
    max_yrmo = 0
    print(f"No existing data found at {sqlite_path}. Starting fresh.")  

Loaded max yrmo: 20251231 from /home/joe/work/Fire/ML/Data/DB/era5_daily_2982_tp.sqlite


In [11]:
hours = [str(hr) for hr in range(0,22,3)]
print(hours)

with sqlite3.connect(sqlite_path) as conn:
    npoints = len(latlons)
    for i, row in latlons.iterrows():
        lat = row['lat']
        lon = row['lon']
        pid = row['point']
        sample = pd.read_sql_query(f"SELECT * FROM {table_name} where point_id = {pid} and datetime > {max_yrmo}", conn)
    #  sample = pd.read_sql_query(f"SELECT * FROM {table_name} where valid_time < '1990-01-04-00'", conn)
        print(f"{i+1}/{npoints} Processing lat {lat}, lon {lon} with {len(sample)} records")  
       
    ##
    ##  Calculate daily means and counts of wind speeds above 20 mph and 25 mph
    ##
        sample['yyyymm'] = sample['datetime'].str[:6]  
        dates=[]
        means=[]
        vars = []
        for gp in sample.groupby('yyyymm'):
            dates.append(gp[0])
            if TARGET_VAR != 'tp':
                a=gp[1][hours].mean()
            
                means.append(a.mean()       )   
            else:
                a=gp[1][hours].sum()
              
                means.append(a.sum()       )   

        dfOut = pd.DataFrame({
            "yrmo": dates,
            "value": means
        })  
        dfOut['point_id'] = pid
        dfOut['latitude'] = lat
        dfOut['longitude'] = lon
        dfOut['var'] = TARGET_VAR

    ##
    ## Calculate monthly means and counts of wind speeds above 20 mph and 25 mph
    ##

        with sqlite3.connect(sqlite_path_out) as conn2:
            dfOut.to_sql(table_name_out, conn2, index=False, if_exists='append')

    

['0', '3', '6', '9', '12', '15', '18', '21']
1/2982 Processing lat 41.1, lon -109.0 with 90 records
2/2982 Processing lat 41.1, lon -108.9 with 90 records
3/2982 Processing lat 41.1, lon -108.8 with 90 records
4/2982 Processing lat 41.1, lon -108.70000000000002 with 90 records
5/2982 Processing lat 41.1, lon -108.60000000000002 with 90 records
6/2982 Processing lat 41.1, lon -108.50000000000004 with 90 records
7/2982 Processing lat 41.1, lon -108.40000000000003 with 90 records
8/2982 Processing lat 41.1, lon -108.30000000000004 with 90 records
9/2982 Processing lat 41.1, lon -108.20000000000005 with 90 records
10/2982 Processing lat 41.1, lon -108.10000000000004 with 90 records
11/2982 Processing lat 41.1, lon -108.00000000000006 with 90 records
12/2982 Processing lat 41.1, lon -107.90000000000006 with 90 records
13/2982 Processing lat 41.1, lon -107.80000000000008 with 90 records
14/2982 Processing lat 41.1, lon -107.70000000000007 with 90 records
15/2982 Processing lat 41.1, lon -107